In [ ]:
# ── Cell 1: Colab GPU Setup ───────────────────────────────────────────────────
# Requires T4 GPU runtime: Runtime → Change runtime type → T4 GPU

import subprocess, sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/fmd'
    IN_COLAB = True
except ImportError:
    PROJECT_DIR = '/Users/aman2394/Desktop/ICWSM-2026-FMD'
    IN_COLAB = False
    print('Running locally — skipping Drive mount')

os.makedirs(f'{PROJECT_DIR}/features', exist_ok=True)
sys.path.insert(0, f'{PROJECT_DIR}/src')

# Install dependencies
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'sentence-transformers', 'datasets',
    'accelerate', 'scikit-learn', 'pandas', 'numpy', 'tqdm', 'scipy'
], check=True)

# Verify GPU
import torch
assert torch.cuda.is_available(), (
    'No GPU detected. Runtime → Change runtime type → T4 GPU'
)
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PROJECT_DIR = {PROJECT_DIR}')

In [ ]:
# ── Cell 2: Load Data (Train Split Only) ─────────────────────────────────────
# Loads train split: original train samples + augmented False samples
# Val and test sets are held out — used only in notebook 06/07.

import sys
sys.path.insert(0, f'{PROJECT_DIR}/src')

import numpy as np
from utils.data_loader import load_combined_data
from utils.data_splitter import get_split_records

orig_records = load_combined_data(PROJECT_DIR)
AUG_PATH = f'{PROJECT_DIR}/data/augmented/augmented_train.json'

train_records, val_records, test_records = get_split_records(
    all_records=orig_records,
    augmented_path=AUG_PATH,
    project_dir=PROJECT_DIR,
)

texts  = [r['text']  for r in train_records]
labels = np.array([int(r['label']) for r in train_records], dtype=np.int32)

n_true  = (labels == 1).sum()
n_false = (labels == 0).sum()
n_aug   = sum(r.get('perturbation_type') is not None for r in train_records)

print(f'Train set  : {len(texts)} samples')
print(f'  True     : {n_true}')
print(f'  False    : {n_false}  (of which {n_aug} are augmented)')
print(f'  Balance  : {labels.mean():.2%} true')
print(f'Val set    : {len(val_records)} samples (held out)')
print(f'Test set   : {len(test_records)} samples (held out)')
print(f'\nSample text: {texts[0][:200]}')


In [ ]:
# ── Cell 3: Extract NLI Internal-Consistency Features ────────────────────────
# Model : cross-encoder/nli-deberta-v3-large
# Output: (N, 5) array — contradiction_ratio, max_contradiction_score,
#         entailment_ratio, coherence_score, weighted_contradiction
# Est.  : ~2–3 hrs on T4

from features.tier1_nli import extract_nli_feature_matrix

print('Extracting NLI features (this takes ~2–3 hrs on T4) ...')
nli_features = extract_nli_feature_matrix(
    texts=texts,
    device='cuda',
    batch_size=32,
    checkpoint_path=f'{PROJECT_DIR}/feature_cache/_nli_checkpoint.npy',
)

print(f'NLI features shape : {nli_features.shape}')   # (N, 5)
print(f'Sample row         : {nli_features[0]}')

# Save checkpoint immediately
import numpy as np
np.save(f'{PROJECT_DIR}/feature_cache/_nli_features.npy', nli_features)
print('NLI features checkpoint saved.')

In [ ]:
# ── Cell 4: Extract FinBERT CLS Embeddings + Distance Features ───────────────
# Model : ProsusAI/finbert
# Output: (N, 2) — cosine_dist + mahalanobis_dist from True-class centroid
# IMPORTANT: saves emb_extractor.pkl — required for blind-set inference

import numpy as np
from features.tier1_embeddings import extract_all_embeddings, EmbeddingDistanceExtractor

print("Extracting FinBERT CLS embeddings...")
embeddings = extract_all_embeddings(texts, device="cuda")
np.save(f"{PROJECT_DIR}/feature_cache/_raw_embeddings.npy", embeddings)
print(f"Raw embeddings shape: {embeddings.shape}")

# Fit extractor on True-class training embeddings, transform all
extractor = EmbeddingDistanceExtractor()
t1_emb = extractor.fit_transform(embeddings, labels)
print(f"Embedding distance features shape: {t1_emb.shape}")

# Save fitted extractor — MUST be loaded at inference time (do not refit on blind data)
import os
os.makedirs(f"{PROJECT_DIR}/models", exist_ok=True)
extractor.save(f"{PROJECT_DIR}/models/emb_extractor.pkl")
np.save(f"{PROJECT_DIR}/feature_cache/_emb_features.npy", t1_emb)
print("✅ Embedding features + extractor saved.")


In [ ]:
# ── Cell 5: Extract Discourse Coherence Features ─────────────────────────────
# Model : sentence-transformers/all-mpnet-base-v2
# Output: (N, 4) array — consecutive_similarity_min, consecutive_similarity_std,
#                         first_last_similarity, mean_coherence
# Est.  : ~5 mins on T4

from features.tier1_coherence import extract_coherence_feature_matrix

print('Extracting discourse coherence features ...')
coherence_features = extract_coherence_feature_matrix(
    texts=texts,
    device='cuda',
    batch_size=64,
)

print(f'Coherence features shape : {coherence_features.shape}')  # (N, 4)
print(f'Sample row               : {coherence_features[0]}')

import numpy as np
np.save(f'{PROJECT_DIR}/feature_cache/_coherence_features.npy', coherence_features)
print('Coherence features checkpoint saved.')

In [ ]:
# ── Cell 6: Extract Epistemic Calibration Features (CPU) ─────────────────────
# Rule-based — no GPU needed
# Output: (N, 3) array — certainty_ratio, hedge_density,
#                         certainty_evidence_mismatch

from features.tier1_epistemic import extract_epistemic_feature_matrix

print('Extracting epistemic calibration features (CPU, rule-based) ...')
epistemic_features = extract_epistemic_feature_matrix(texts=texts)

print(f'Epistemic features shape : {epistemic_features.shape}')  # (N, 3)
print(f'Sample row               : {epistemic_features[0]}')

import numpy as np
np.save(f'{PROJECT_DIR}/feature_cache/_epistemic_features.npy', epistemic_features)
print('Epistemic features checkpoint saved.')

In [ ]:
# ── Cell 7: Concatenate All Tier 1 Features → Save ───────────────────────────

import numpy as np

# Load from checkpoints if cells were run in separate sessions
nli_features       = np.load(f'{PROJECT_DIR}/feature_cache/_nli_features.npy')
emb_features       = np.load(f'{PROJECT_DIR}/feature_cache/_emb_features.npy')
coherence_features = np.load(f'{PROJECT_DIR}/feature_cache/_coherence_features.npy')
epistemic_features = np.load(f'{PROJECT_DIR}/feature_cache/_epistemic_features.npy')

print('Individual feature shapes:')
print(f'  NLI        : {nli_features.shape}')
print(f'  Embeddings : {emb_features.shape}')
print(f'  Coherence  : {coherence_features.shape}')
print(f'  Epistemic  : {epistemic_features.shape}')

# Concatenate horizontally
tier1_features = np.hstack([
    nli_features,
    emb_features,
    coherence_features,
    epistemic_features,
])

print(f'\nTier 1 combined shape : {tier1_features.shape}')

# Save final arrays
np.save(f'{PROJECT_DIR}/feature_cache/tier1_features.npy', tier1_features)
np.save(f'{PROJECT_DIR}/feature_cache/labels.npy', labels)

print('✅ Saved:')
print(f'  {PROJECT_DIR}/feature_cache/tier1_features.npy  — shape {tier1_features.shape}')
print(f'  {PROJECT_DIR}/feature_cache/labels.npy           — shape {labels.shape}')

# Quick sanity check: are NaN-free?
assert not np.isnan(tier1_features).any(), 'NaN values detected in tier1_features!'
print('NaN check passed.')